In [1]:
# =========================
# Regression Metrics in Python
# =========================
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Example true vs. predicted values
y_true = np.array([3.0, -0.5, 2.0, 7.0])
y_pred = np.array([2.5,  0.0, 2.0, 8.0])

# Number of samples and predictors
n = len(y_true)
p = 1  # e.g., one predictor in the model

# 1. Mean Squared Error
mse  = mean_squared_error(y_true, y_pred)

# 2. Root MSE
rmse = np.sqrt(mse)

# 3. Mean Absolute Error
mae  = mean_absolute_error(y_true, y_pred)

# 4. Mean Absolute Percentage Error
mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100

# 5. R2
r2 = r2_score(y_true, y_pred)

# 6. Adjusted R2
adj_r2  = 1 - (1 - r2) * (n - 1) / (n - p - 1)

In [2]:
# =========================
# Classification Metrics in Python
# =========================
import numpy as np
from sklearn.metrics import (
    accuracy_score, precision_score,
    recall_score, f1_score, confusion_matrix
)

# Example true & predicted labels
y_true = np.array([1, 0, 1, 1, 0, 0, 1, 0])
y_pred = np.array([1, 0, 0, 1, 1, 0, 1, 0])

# Compute basic accuracy, precision, recall, F1
accuracy  = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred)
recall    = recall_score(y_true, y_pred)       # same as TPR
f1        = f1_score(y_true, y_pred)

# Compute specificity (TNR) from confusion matrix
tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
specificity = tn / (tn + fp)

# Alternate way: treat the 'negative' class (0) as the 'positive' class
specificity = recall_score(y_true, y_pred, pos_label=0)

In [3]:
# =========================
# Classification Curves & AUC in Python
# =========================
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import (
    roc_curve, auc,
    precision_recall_curve, average_precision_score
)

# Example true labels and prediction scores
y_true  = np.array([0,1,1,0,1,0,1,0,1,0])
y_score = np.array([0.1,0.8,0.7,0.3,0.9,0.2,0.6,0.4,0.85,0.05])

# 1. ROC Curve & AUC
fpr, tpr, _ = roc_curve(y_true, y_score)
roc_auc     = auc(fpr, tpr)

# 2. Precision-Recall Curve & AUPRC
precision, recall, _ = precision_recall_curve(y_true, y_score)
pr_auc              = average_precision_score(y_true, y_score)

In [4]:
# =========================
# Little’s MCAR Test from Scratch
# =========================
import numpy as np
import pandas as pd
from scipy.stats import chi2

def little_mcar_test(df):
    # Copy and identify missingness patterns
    df0 = df.copy()
    df0['pattern'] = df0.isnull().apply(lambda row: tuple(row), axis=1)
    # Covariance from complete cases
    cov_complete = df0.drop(columns='pattern').dropna().cov()
    test_stat = 0.0
    df_deg   = 0
    # For each pattern, compute contribution
    for pat, group in df0.groupby('pattern'):
        obs_cols = [col for col, miss in zip(df0.columns[:-1], pat) if not miss]
        n_i = len(group)
        if n_i < 1 or len(obs_cols) < 2:
            continue
        # Sub-covariance and its inverse
        S_obs = cov_complete.loc[obs_cols, obs_cols].values
        S_inv = np.linalg.inv(S_obs)
        # Means
        mu_i = group[obs_cols].mean().values
        mu   = df0[obs_cols].mean().values
        diff = mu_i - mu
        # Accumulate statistic and degrees of freedom
        test_stat += n_i * diff.dot(S_inv.dot(diff))
        df_deg   += len(obs_cols) - 1
    # p‐value from chi‐square
    p_value = 1 - chi2.cdf(test_stat, df_deg)
    return test_stat, df_deg, p_value

In [6]:
pip install pyampute

In [ ]:
# =========================
# Little’s MCAR Test with pyampute
# =========================
import pandas as pd
from pyampute.exploration.mcar_statistical_tests import MCARTest

# Load dataset with missing values
df = pd.read_csv('data_with_missing.csv')

# Perform Little's MCAR test
mt = MCARTest(method='little')
p_value = mt.little_mcar_test(df)

In [ ]:
# =========================
# Simple Imputation in Python
# =========================
import pandas as pd

# Load example DataFrame with missing values
df = pd.read_csv('data_with_missing.csv')

# 1. Mean Imputation
df_mean = df.copy()
for col in df_mean.select_dtypes(include='number'):
    mu = df_mean[col].mean()
    df_mean[col].fillna(mu, inplace=True)

# 2. Median Imputation
df_med = df.copy()
for col in df_med.select_dtypes(include='number'):
    m = df_med[col].median()
    df_med[col].fillna(m, inplace=True)

# 3. Forward / Backward Fill
df_ffill = df.copy()
df_ffill.fillna(method='ffill', inplace=True)  # forward fill
df_bfill = df.copy()
df_bfill.fillna(method='bfill', inplace=True)  # backward fill

# 4. Missingness Indicator
df_ind = df.copy()
for col in df_ind.columns:
    if df_ind[col].isnull().any():
        df_ind[col + '_miss'] = df_ind[col].isnull().astype(int)

In [ ]:
# =========================
# Model-Based Imputation: Regression & k-NN
# =========================
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.impute import KNNImputer

# Load DataFrame with missing values
df = pd.read_csv('data_with_missing.csv')

# 1. Regression Imputation
df_reg = df.copy()
for target in df_reg.columns:
    if not df_reg[target].isnull().any():
        continue
    train = df_reg[df_reg[target].notnull()].dropna(axis=1, how='all')
    X_train = train.drop(columns=[target])
    y_train = train[target]
    predict = df_reg[df_reg[target].isnull()]
    X_pred = predict.drop(columns=[target])
    model = LinearRegression()
    model.fit(X_train, y_train)
    df_reg.loc[df_reg[target].isnull(), target] = model.predict(X_pred)

# 2. k-NN Imputation
df_knn = df.copy()
imputer = KNNImputer(n_neighbors=5, weights="uniform")
imputed_array = imputer.fit_transform(df_knn)
df_knn = pd.DataFrame(imputed_array, columns=df_knn.columns)

In [ ]:
# =========================
# Multiple Imputation (MICE) in Python
# =========================
import pandas as pd, numpy as np
from sklearn.experimental import enable_iterative_imputer  # noqa
from sklearn.impute import IterativeImputer
from sklearn.linear_model import LinearRegression

# 1. Generate m imputed datasets
m = 5
imputed_dfs = []
for seed in range(m):
    imp = IterativeImputer(sample_posterior=True, random_state=seed)
    imputed = imp.fit_transform(df)
    imputed_dfs.append(pd.DataFrame(imputed, columns=df.columns))

# 2. Fit model on each dataset and collect estimates & variances
thetas, Us = [], []
for imp_df in imputed_dfs:
    X = imp_df.drop(columns='target').values
    y = imp_df['target'].values
    model = LinearRegression().fit(X, y)
    theta = model.coef_
    thetas.append(theta)
    preds = model.predict(X)
    resid = y - preds
    sigma2 = np.var(resid, ddof=X.shape[1])
    cov = sigma2 * np.linalg.inv(X.T @ X)
    Us.append(np.diag(cov))

# 3. Pool estimates
thetas = np.array(thetas)
Us     = np.array(Us)
theta_bar = thetas.mean(axis=0)
B         = thetas.var(axis=0, ddof=1)
U_bar     = Us.mean(axis=0)
T         = U_bar + (1 + 1/m) * B

In [ ]:
# =========================
# Outlier Detection: IQR Rule & Isolation Forest
# =========================
import pandas as pd
from sklearn.ensemble import IsolationForest

# Load DataFrame
df = pd.read_csv('data.csv')

# 1. IQR Rule
Q1 = df.quantile(0.25)
Q3 = df.quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outlier_mask = ((df < lower_bound) | (df > upper_bound)).any(axis=1)
clean_iqr = df[outlier_mask == False]
outliers_iqr = df[outlier_mask]

# 2. Isolation Forest
iso = IsolationForest(contamination=0.05, random_state=0)
iso.fit(df)
labels = iso.predict(df)  # 1 = inlier, -1 = outlier

outliers_if = df[labels == -1]
clean_if    = df[labels == 1]

In [ ]:
# =========================
# Outlier Handling: Winsorizing & Robust Scaling
# =========================
import pandas as pd
from scipy.stats.mstats import winsorize
from sklearn.preprocessing import RobustScaler

# Load DataFrame
df = pd.read_csv('data.csv')

numeric_cols = df.select_dtypes(include='number').columns

# 1. Winsorizing
p = 0.01
df_wins = df.copy()
df_wins[numeric_cols] = df[numeric_cols].apply(
    lambda col: winsorize(col, limits=(p, p))
)

# 2. Robust Scaling
scaler = RobustScaler()
df_robust = df.copy()
df_robust[numeric_cols] = scaler.fit_transform(df[numeric_cols])

In [ ]:
# =========================
# Numerical Feature Transformations in Python
# =========================
import pandas as pd
import numpy as np
from sklearn.preprocessing import PowerTransformer

# Load DataFrame
df = pd.read_csv('data.csv')

numeric_cols = df.select_dtypes(include='number').columns

# 1. Log Transform
epsilon = 1e-6
df_log = df.copy()
df_log[numeric_cols] = np.log(df[numeric_cols] + epsilon)

# 2. Box-Cox Transform
df_bc = df.copy()
pt = PowerTransformer(method='box-cox')
df_bc[numeric_cols] = pt.fit_transform(df[numeric_cols] + epsilon)

lambdas = pd.Series(pt.lambdas_, index=numeric_cols)
print("Box-Cox lambdas per feature:\n", lambdas)

In [ ]:
# =========================
# Categorical Encoding in Python
# =========================
import pandas as pd

# Assume df has categorical column 'cat_col' and numeric target 'target'

# 1. One-Hot Encoding
df_ohe = pd.get_dummies(df, columns=['cat_col'], prefix='cat')

# 2. Target Encoding (smoothed)
alpha = 10
global_mean = df['target'].mean()
stats = df.groupby('cat_col')['target'].agg(count='count', mean='mean')
stats['te'] = (stats['mean'] * stats['count'] +
               global_mean * alpha) / (stats['count'] + alpha)

df_te = df.copy()
df_te['cat_col_te'] = df_te['cat_col'].map(stats['te'])

In [ ]:
# =========================
# Feature Scaling in Python
# =========================
import pandas as pd
from sklearn.preprocessing import MinMaxScaler, StandardScaler

# Load DataFrame
df = pd.read_csv('data.csv')

numeric_cols = df.select_dtypes(include='number').columns

# 1. Min-Max Normalization
scaler_mm = MinMaxScaler()
df_mm = df.copy()
df_mm[numeric_cols] = scaler_mm.fit_transform(df[numeric_cols])

# 2. Standardization
scaler_std = StandardScaler()
df_std = df.copy()
df_std[numeric_cols] = scaler_std.fit_transform(df[numeric_cols])